<a href="https://colab.research.google.com/github/Afvalland/PPI-/blob/main/RFdiffusion3_design_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RFdiffusion3 design pipeline (Foundry)

General-purpose **RFD3 → LigandMPNN → RF3** pipeline on Colab. The input JSON builder is
parameterised, so the same notebook covers unconditional generation, monomer/motif
scaffolding, binder design, and **Cn/Dn symmetric** design — set the parameters for the run
you want.

**Stack** — [RosettaCommons/foundry](https://github.com/RosettaCommons/foundry):
`rfd3` (all-atom diffusion) → `mpnn` (LigandMPNN inverse folding) → `rf3` (open AF3-class validator).

**Before you run anything**
- `Runtime → Change runtime type → GPU`. RFD3 + RF3 are heavy. A **T4 (16 GB) is the floor**
  (use `low_memory_mode`); an **L4 / A100** is much more comfortable. Symmetric runs are
  forced to `diffusion_batch_size=1`, so watch VRAM on large symmetric targets.
- Install + weights take a while (multi-GB). Do them once per session.

**Scope note.** Install, MPNN, and RF3 follow the documented CLI and are mechanical. The
parts that need empirical iteration are the **contig + hotspot** spec for your target and —
if you go symmetric — preparing an **exactly Cn/Dn, origin-centred, axis-aligned** target
(step 5). The JSON builder prints the exact spec so you can hand-edit it.


## 0 依赖版本匹配调整

In [1]:
#先把 AtomWorks 固定到 Foundry 支持范围的最低版本 3.0.1 试试
!pip install -q --force-reinstall --no-deps "atomworks==3.0.1"

## 1 · Check the GPU

In [2]:
#@title Check allocated GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "No GPU — set Runtime → Change runtime type → GPU"


name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


## Setup · Run name, output directory & Google Drive

Name this run (the **binder name**) and mount Drive. Each run gets its own timestamped
folder — `<BASE_DIR>/runs/<date-time>_<RUN_NAME>` — so runs never overwrite each other, and
the `WORK` variable every downstream step uses points at it automatically.

**Resuming an interrupted campaign:** set `RESUME_RUN` to `latest`, to a run folder name, or
to a full path, and this cell reattaches to that run instead of starting a new one. Leave it
empty for a new run. Colab disconnects mid-campaign are the normal reason to use it — after a
reconnect, run this cell with `RESUME_RUN = latest`, then steps 2–3, then carry on.

Model weights are stored once under `<BASE_DIR>/foundry_ckpts` and reused by every future
run (see step 3), so you only download them the first time.

In [3]:
#@title Mount Drive, name this run & set output directory
import os, re, glob, datetime
MOUNT_DRIVE = True                            #@param {type:"boolean"}
BASE_DIR    = "/content/drive/MyDrive/RFD3"   #@param {type:"string"}
RUN_NAME    = "1SHQ_test_02_261026"                         #@param {type:"string"}
RESUME_RUN  = ""                               #@param {type:"string"}
#  RESUME_RUN: "" = start a new run.  "latest" = reattach to the most recent run.
#  Or give a run folder name ("20260712-191003_mybinder") or a full path.

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

# Shared, persistent checkpoints (downloaded once, reused by every future run)
CKPT_DIR = f"{BASE_DIR}/foundry_ckpts"
os.environ["FOUNDRY_CHECKPOINT_DIRS"] = CKPT_DIR
os.makedirs(CKPT_DIR, exist_ok=True)

RUNS_DIR = f"{BASE_DIR}/runs"
os.makedirs(RUNS_DIR, exist_ok=True)

if RESUME_RUN.strip():                                    # ---- resume an existing run ----
    r = RESUME_RUN.strip().rstrip("/")
    if r.lower() == "latest":
        cands = sorted(d for d in glob.glob(f"{RUNS_DIR}/*") if os.path.isdir(d))
        assert cands, f"No runs found under {RUNS_DIR} to resume."
        WORK = cands[-1]                                  # timestamp prefix sorts chronologically
    else:
        WORK = r if os.path.isabs(r) else f"{RUNS_DIR}/{r}"
    assert os.path.isdir(WORK), f"Run folder not found: {WORK}"
    RUN_NAME = re.sub(r"^\d{8}-\d{6}_", "", os.path.basename(WORK))
    print("RESUMED run        :", WORK)
else:                                                     # ---- start a new run ----
    # sanitize: spaces and other characters break the Hydra command line in step 8
    clean = re.sub(r"[^A-Za-z0-9._-]", "_", RUN_NAME.strip()) or "binder"
    if clean != RUN_NAME:
        print(f"RUN_NAME sanitized: {RUN_NAME!r} -> {clean!r}")
    RUN_NAME = clean
    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    WORK = f"{RUNS_DIR}/{stamp}_{RUN_NAME}"
    os.makedirs(WORK, exist_ok=True)
    print("NEW run            :", WORK)

print("Checkpoints (shared):", CKPT_DIR)
print("Run name            :", RUN_NAME)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NEW run            : /content/drive/MyDrive/RFD3/runs/20261026-130058_1SHQ_test_02_261026
Checkpoints (shared): /content/drive/MyDrive/RFD3/foundry_ckpts
Run name            : 1SHQ_test_02_261026


## Setup · Campaign status  *(run any time)*

Reports what this run has already produced and what is still outstanding at each stage, so an
interrupted campaign can be picked up where it stopped. Re-run it after any step to see progress.

With `SKIP_EXISTING` left on in steps 8–10, simply re-running those steps tops the campaign up
rather than redoing finished work.

In [4]:
#@title Campaign status — what's done, what's left
import os, glob
WORK = globals().get("WORK", "/content/run")

RFD3_OUT, MPNN_OUT, RF3_OUT = f"{WORK}/rfd3_out", f"{WORK}/mpnn_out", f"{WORK}/rf3_out"
backbones = sorted(glob.glob(f"{RFD3_OUT}/**/*.cif.gz", recursive=True))
mpnn_cifs = sorted(glob.glob(f"{MPNN_OUT}/**/*.cif", recursive=True))
rf3_files = [p for p in glob.glob(f"{RF3_OUT}/**/*", recursive=True) if os.path.isfile(p)]

def _has_output(stem, names):
    for n in names:
        if n.startswith(stem) and (len(n) == len(stem) or not n[len(stem)].isdigit()):
            return True
    return False

# backbone X.cif.gz -> MPNN writes X.cif_b0_dN.cif ; MPNN Y.cif -> RF3 writes files starting Y
mpnn_names = [os.path.basename(p) for p in mpnn_cifs]
rf3_names  = [os.path.basename(p) for p in rf3_files]
bb_done = [b for b in backbones
           if any(n.startswith(os.path.basename(b)[:-3] + "_") for n in mpnn_names)]
mp_done = [m for m in mpnn_cifs if _has_output(os.path.basename(m)[:-4], rf3_names)]

print("Run:", WORK, "\n")
print(f"  input.json          : {'yes' if os.path.exists(f'{WORK}/input.json') else 'MISSING — run step 7'}")
print(f"  8  RFD3 backbones   : {len(backbones)}")
print(f"  9  MPNN  designed   : {len(mpnn_cifs)} sequences from {len(bb_done)}/{len(backbones)} backbones")
print(f"  10 RF3   refolded   : {len(mp_done)}/{len(mpnn_cifs)} designs")
print(f"  11 metrics csv      : {'yes' if os.path.exists(f'{WORK}/design_metrics.csv') else 'no'}")
fa = f"{MPNN_OUT}/binder_sequences.fasta"
print(f"     binder FASTA     : {'yes' if os.path.exists(fa) else 'no'}")

print("\nNext:")
if not backbones:
    print("  -> step 8 (no backbones yet)")
elif len(bb_done) < len(backbones):
    print(f"  -> step 9 ({len(backbones)-len(bb_done)} backbone(s) still need sequences)")
elif len(mp_done) < len(mpnn_cifs):
    print(f"  -> step 10 ({len(mpnn_cifs)-len(mp_done)} design(s) still need refolding)")
else:
    print("  -> step 11 (all stages complete; add more backbones by re-running step 8)")


Run: /content/drive/MyDrive/RFD3/runs/20261026-130058_1SHQ_test_02_261026 

  input.json          : MISSING — run step 7
  8  RFD3 backbones   : 0
  9  MPNN  designed   : 0 sequences from 0/0 backbones
  10 RF3   refolded   : 0/0 designs
  11 metrics csv      : no
     binder FASTA     : no

Next:
  -> step 8 (no backbones yet)


## 2 · Install Foundry

`rc-foundry[all]` pulls RFD3, LigandMPNN, RF3 and the shared AtomWorks stack. This is the
slow cell. If the PyPI install hits a resolver/CUDA snag on Colab, fall back to the git
install in the commented block.


In [5]:
#@title Install rc-foundry  (~several minutes)
import time, os
t0 = time.time()

# Primary path: PyPI
#!pip install -q "rc-foundry[all]"

# --- Fallback (uncomment if the above fails) ----------------------------------
!git clone -q https://github.com/RosettaCommons/foundry.git
!pip install -q -e "foundry/.[all]"
# ------------------------------------------------------------------------------

# Light deps used by the prep/analysis cells below
!pip install -q biotite pandas

print(f"\nInstall finished in {time.time()-t0:.0f}s")
# Sanity check the CLIs are on PATH
!which rfd3 mpnn rf3 foundry || true


fatal: destination path 'foundry' already exists and is not an empty directory.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for rc-foundry (pyproject.toml) ... done

Install finished in 12s
/usr/local/bin/rfd3
/usr/local/bin/mpnn
/usr/local/bin/rf3
/usr/local/bin/foundry


## 3 · Download model weights  *(once)*

Downloads RFD3 / LigandMPNN / RF3 into the shared `foundry_ckpts` folder on Drive. The cell
checks whether they're already there and **skips the download if so**, so on every run after
the first this step is instant.

In [6]:
#@title Download model weights  (once; cached on Drive for future runs)
import os, glob
CKPT_DIR = os.environ.get("FOUNDRY_CHECKPOINT_DIRS", "/content/foundry_ckpts")
os.makedirs(CKPT_DIR, exist_ok=True)

present = {k: bool(glob.glob(f"{CKPT_DIR}/*{k}*")) for k in ["rfd3", "mpnn", "rf3"]}
if all(present.values()):
    print("All checkpoints already present in", CKPT_DIR, "\u2014 skipping download.")
else:
    print("Missing:", [k for k, v in present.items() if not v], "\u2014 downloading...")
    get_ipython().system(f'foundry install rfd3 ligandmpnn rf3 --checkpoint-dir "{CKPT_DIR}"')

print("\nCheckpoints:")
get_ipython().system(f'ls -lh "{CKPT_DIR}"')


All checkpoints already present in /content/drive/MyDrive/RFD3/foundry_ckpts — skipping download.

Checkpoints:
total 5.4G
-rw------- 1 root root  11M Oct 26 12:46 ligandmpnn_v_32_010_25.pt
-rw------- 1 root root 2.9G Oct 26 12:47 rf3_foundry_01_24_latest_remapped.ckpt
-rw------- 1 root root 2.6G Oct 26 12:46 rfd3_latest.ckpt


## 4 · Provide the target structure *(optional)*

Skip this for unconditional generation. Otherwise upload a PDB/CIF, **or** fetch one by
accession. For binder/motif work, trim the input to the region you actually want to build
against — smaller inputs are kinder on VRAM and easier to reason about.


In [7]:
#@title Load target (upload or fetch)
import os, urllib.request
SOURCE = "fetch_pdb"  #@param ["upload", "fetch_pdb"]
PDB_ID = "1SHQ"    #@param {type:"string"}
WORK = globals().get("WORK", "/content/run"); os.makedirs(WORK, exist_ok=True)
RAW_TARGET = os.path.join(WORK, "target_raw.pdb")

if SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    src = list(up.keys())[0]
    with open(RAW_TARGET, "wb") as f:   # write bytes directly — works across local/Drive
        f.write(up[src])
else:
    url = f"https://files.rcsb.org/download/{PDB_ID.upper()}.pdb"
    urllib.request.urlretrieve(url, RAW_TARGET)

print("Target ->", RAW_TARGET)
import biotite.structure.io as strucio
arr = strucio.load_structure(RAW_TARGET)
import numpy as np
print("chains:", sorted(set(arr.chain_id)), "| residues:",
      len(np.unique(np.stack([arr.chain_id, arr.res_id]), axis=1).T))


Target -> /content/drive/MyDrive/RFD3/runs/20261026-130058_1SHQ_test_02_261026/target_raw.pdb
chains: [np.str_('A'), np.str_('B')] | residues: 1066


## 5 · Symmetric targets only — align axis → z, centre → origin *(skip for non-symmetric runs)*

**Only run this if you're doing symmetric design.** RFD3 symmetric sampling expects the
symmetric motif **pre-symmetrised around the origin** with the symmetry axis on **z**. A raw
cryo-EM/AF3 assembly is usually only approximately symmetric and sits at an arbitrary
position/orientation.

Strategy below:
1. Estimate the Cn axis from the per-chain Cα centres of mass (SVD plane-normal).
2. Translate the ring centre to the origin and rotate the axis onto z.
3. *(Recommended)* Rebuild an **idealised Cn assembly** from a single protomer by applying
   `k·(360/n)°` rotations about z — this guarantees exact symmetry, which RFD3 wants.

Set `RING_CHAINS` to the chains forming your ring, `PROTOMER_CHAIN` to the one you trust most
(best density / highest pLDDT), and `SYM_ORDER` to n. Writes `target_sym.pdb`.


In [ ]:
#@title Align & idealise the symmetric target
import numpy as np, biotite.structure as struc, biotite.structure.io as strucio

RING_CHAINS    = "A,B,C,D,E,F"  #@param {type:"string"}
PROTOMER_CHAIN = "A"            #@param {type:"string"}
SYM_ORDER      = 6              #@param {type:"integer"}
IDEALISE       = True           #@param {type:"boolean"}
ALIGNED_TARGET = f"{globals().get('WORK','/content/run')}/target_sym.pdb"

ring = [c.strip() for c in RING_CHAINS.split(",") if c.strip()]
arr  = strucio.load_structure(f"{globals().get('WORK','/content/run')}/target_raw.pdb")

# per-chain Cα COM
_aa = struc.filter_amino_acids(arr)        # exclude Ca2+ ions, whose atom_name is also "CA"
def chain_ca_com(a, ch):
    m = (a.chain_id == ch) & (a.atom_name == "CA") & _aa
    assert m.sum() > 0, f"No protein CA atoms in chain {ch}"
    return a.coord[m].mean(0)
coms = np.stack([chain_ca_com(arr, c) for c in ring])
centre = coms.mean(0)

# fold-preservation tripwire: helix fraction of the input protomer (rotation-invariant)
def _helix_frac(a):
    try: return float((struc.annotate_sse(a) == "a").mean())
    except Exception: return float("nan")
_src_helix = _helix_frac(arr[arr.chain_id == PROTOMER_CHAIN])

# Cn axis = plane normal of the COM ring (smallest singular vector)
_, _, Vt = np.linalg.svd(coms - centre)
axis = Vt[-1] / np.linalg.norm(Vt[-1])

def rot_a_to_b(a, b):
    # Robust axis-angle rotation aligning unit a onto unit b. Uses arccos + normalized
    # axis so it is stable for ALL angles, including (near-)antiparallel where the older
    # cross-product form divides by (1+c) ~ 0 and blows up (det != 1, corrupts the fold).
    a = a/np.linalg.norm(a); b = b/np.linalg.norm(b)
    c = float(np.clip(np.dot(a, b), -1.0, 1.0))
    if c > 1 - 1e-9:                                  # already aligned
        return np.eye(3)
    if c < -1 + 1e-9:                                 # (near-)antiparallel
        axis = np.cross(a, np.array([1., 0., 0.]))
        if np.linalg.norm(axis) < 1e-6:
            axis = np.cross(a, np.array([0., 1., 0.]))
    else:
        axis = np.cross(a, b)
    axis = axis/np.linalg.norm(axis)
    ang = np.arccos(c)
    K = np.array([[0,-axis[2],axis[1]],[axis[2],0,-axis[0]],[-axis[1],axis[0],0]])
    return np.eye(3) + np.sin(ang)*K + (1 - np.cos(ang))*(K@K)

R = rot_a_to_b(axis, np.array([0.,0.,1.]))
assert np.linalg.det(R) > 0, ("Alignment produced a REFLECTION (det<0), which would invert "
    "the fold. This is the antiparallel-axis bug — update rot_a_to_b.")
arr.coord = (arr.coord - centre) @ R.T          # centre + align axis to z

def rotz(deg):
    t = np.deg2rad(deg); c,s = np.cos(t), np.sin(t)
    return np.array([[c,-s,0],[s,c,0],[0,0,1.]])

if IDEALISE:
    proto = arr[arr.chain_id == PROTOMER_CHAIN].copy()
    labels = [chr(ord("A")+i) for i in range(SYM_ORDER)]
    subs = []
    for i in range(SYM_ORDER):
        s = proto.copy()
        s.coord = s.coord @ rotz(360.0/SYM_ORDER*i).T
        s.chain_id = np.full(s.array_length(), labels[i])
        subs.append(s)
    out = subs[0]
    for s in subs[1:]:
        out += s
else:
    out = arr[np.isin(arr.chain_id, ring)].copy()

strucio.save_structure(ALIGNED_TARGET, out)
_out_helix = _helix_frac(out[out.chain_id == sorted(set(out.chain_id))[0]])
print("Wrote", ALIGNED_TARGET, "| chains:", sorted(set(out.chain_id)))
print("Cn axis (raw frame):", np.round(axis,3), "| ring centre:", np.round(centre,2))
print(f"det(R) = {np.linalg.det(R):+.2f} (must be +1)  |  helix fraction in:{_src_helix:.2f} out:{_out_helix:.2f}")
if _src_helix == _src_helix and _out_helix == _out_helix and _src_helix > 0.10 and _out_helix < 0.5*_src_helix:
    print("\u26a0\ufe0f  FOLD COLLAPSED during idealisation (helix content dropped sharply).")
    print("    Do NOT use this output. The structure was corrupted; check the alignment step.")
else:
    print("\u2713 fold preserved.")


## 6 · Prepare target — strip ligands / trim residues  *(do this for large complexes)*

This is the cell that keeps RFD3 in memory. The model featurises **everything** in the input
(atomised ligands and full-length chains included), and pairwise cost scales as O(N²) in tokens,
so a full multi-protomer assembly with bound nucleotide will OOM even on an A100. Trim the input
to the chains and residue window you actually design against, and drop non-polymer atoms (ATP,
ions, waters) unless you specifically need them.

- **Symmetric run:** keep all ring chains (`A,B,C,D,E,F`) and trim to your epitope window — that
  gives the small pre-symmetrised motif the C6 sampler replicates.
- **Asymmetric / multi-protomer run:** keep just the 2–3 protomers around your site, trimmed.
- **Keeping a ligand:** leave `STRIP_NONPOLYMER` on for now; the clean way to retain a nucleotide
  is via RFD3's `ligand` field, not by letting it ride along as unnamed hetero.


In [8]:
#@title Prepare / trim target
import os, numpy as np, biotite.structure as struc, biotite.structure.io as strucio

SOURCE_PDB       = "auto"             #@param {type:"string"}
KEEP_CHAINS      = "B"      #@param {type:"string"}
RES_LO           = 0                   #@param {type:"integer"}
RES_HI           = 0                   #@param {type:"integer"}
STRIP_NONPOLYMER = False                #@param {type:"boolean"}
# SOURCE_PDB "auto" = target_sym.pdb if it exists (symmetric), else target_raw.pdb
# RES_LO/RES_HI both 0 = no residue trim. KEEP_CHAINS "" = all chains.

WORK = globals().get("WORK", "/content/run")
if SOURCE_PDB == "auto":
    SOURCE_PDB = (f"{WORK}/target_sym.pdb" if os.path.exists(f"{WORK}/target_sym.pdb")
                  else f"{WORK}/target_raw.pdb")
PREPARED = f"{WORK}/target_prepared.pdb"

arr  = strucio.load_structure(SOURCE_PDB)
mask = np.ones(arr.array_length(), bool)
if STRIP_NONPOLYMER:
    mask &= struc.filter_amino_acids(arr)          # drops ATP / ions / waters
if KEEP_CHAINS.strip():
    mask &= np.isin(arr.chain_id, [c.strip() for c in KEEP_CHAINS.split(",") if c.strip()])
if RES_LO and RES_HI:
    mask &= (arr.res_id >= RES_LO) & (arr.res_id <= RES_HI)

arr = arr[mask]
# 按链和残基编号排序，避免 RFD3 读取金属等非蛋白原子时报顺序错误
order = np.lexsort((arr.atom_name, arr.res_id, arr.chain_id))
arr = arr[order]

strucio.save_structure(PREPARED, arr)
n_res = len(np.unique(np.stack([arr.chain_id, arr.res_id]), axis=1).T)
print(f"Source : {SOURCE_PDB}")
print(f"Wrote  : {PREPARED}")
print(f"Kept   : {arr.array_length()} atoms | {n_res} residues | chains {sorted(set(arr.chain_id))}")
print("\nUse this path as INPUT_PDB in the next cell.")


Source : /content/drive/MyDrive/RFD3/runs/20261026-130058_1SHQ_test_02_261026/target_raw.pdb
Wrote  : /content/drive/MyDrive/RFD3/runs/20261026-130058_1SHQ_test_02_261026/target_prepared.pdb
Kept   : 3809 atoms | 533 residues | chains [np.str_('B')]

Use this path as INPUT_PDB in the next cell.


In [9]:
import biotite.structure as struc
import biotite.structure.io as strucio
import numpy as np

arr = strucio.load_structure(PREPARED)
for name in ["ZN", "MG", "SO4", "NAG"]:
    mask = arr.res_name == name
    print(name, "atoms:", int(mask.sum()),
          "chains:", sorted(set(arr.chain_id[mask])) if mask.any() else [])

ZN atoms: 2 chains: [np.str_('B')]
MG atoms: 1 chains: [np.str_('B')]
SO4 atoms: 10 chains: [np.str_('B')]
NAG atoms: 14 chains: [np.str_('B')]


## 7 · Configure the design run

Build the RFD3 input JSON. Common patterns
([input spec](https://rosettacommons.github.io/foundry/models/rfd3/input.html),
[symmetry](https://rosettacommons.github.io/foundry/models/rfd3/examples/symmetry.html),
[PPI tutorial](https://rosettacommons.github.io/foundry/models/rfd3/tutorials/ppi_design_tutorial.html)):

- **Unconditional / monomer** — leave `INPUT_PDB` empty and set `BINDER_LENGTH` to a single
  number or range (the spec falls back to `length`-only with no input).
- **Binder / motif scaffolding** — point `INPUT_PDB` at your prepared target
  (`/content/run/target_prepared.pdb` from step 6), set `CONTIG` to keep the
  target (`<chain><range>`), a chain break (`/0`), then a designed segment (e.g. `60-100`).
  Add `select_hotspots` to steer the interface (atoms ≤ ~4.5 Å; residues must appear in the contig).
- **Symmetric** — set `USE_SYMMETRY=True` and `SYM_ID` (e.g. `C6`, `D2`; only C/D supported),
  point `INPUT_PDB` at the prepared symmetric file from step 6, and write `CONTIG` as one
  asymmetric unit (RFD3 replicates it n-fold).

`infer_ori_strategy: hotspots` places the design COM ~10 Å out from the hotspot COM;
`is_non_loopy: true` biases toward helices (good for PPI). The cell prints the JSON for
hand-editing.


In [10]:
#@title Build input.json — SAP binder with metal cofactors
import json
import os
import numpy as np
import biotite.structure as struc
import biotite.structure.io as strucio

DESIGN_NAME = globals().get("RUN_NAME", "SAP_binder")

WORK = globals().get("WORK", "/content/run")
INPUT_PDB = f"{WORK}/target_prepared.pdb"
TARGET_CHAIN = "B"             #@param {type:"string"}
BINDER_LENGTH = "60-100"       #@param {type:"string"}
HOTSPOTS = "B120,B122,B123,B160,B172"                  #@param {type:"string"}
LIGANDS = "ZN,MG"              #@param {type:"string"}
IS_NON_LOOPY = True            #@param {type:"boolean"}
INFER_ORI = "hotspots"         #@param ["hotspots", "com", "none"]
USE_SYMMETRY = False           #@param {type:"boolean"}
SYM_ID = "C2"                  #@param {type:"string"}

# 读取 Step 6 的输出，并检查目标链、金属和热点设置
if not os.path.exists(INPUT_PDB):
    raise FileNotFoundError(
        f"找不到输入结构：{INPUT_PDB}\n"
        "请确认 Step 6 已重新运行，且 STRIP_NONPOLYMER=False。"
    )

arr = strucio.load_structure(INPUT_PDB)

protein_mask = struc.filter_amino_acids(arr)
chain_mask = arr.chain_id == TARGET_CHAIN
target_res_ids = np.unique(arr.res_id[protein_mask & chain_mask])

if len(target_res_ids) == 0:
    raise ValueError(
        f"结构中没有找到链 {TARGET_CHAIN} 的蛋白质残基。"
        "请检查 TARGET_CHAIN 和 Step 6 的 KEEP_CHAINS。"
    )

present_names = set(str(x).strip() for x in arr.res_name)
requested_ligands = [x.strip() for x in LIGANDS.split(",") if x.strip()]
missing_ligands = [name for name in requested_ligands if name not in present_names]

if missing_ligands:
    raise ValueError(
        f"输入结构中找不到这些配体/金属组分：{missing_ligands}\n"
        "请确认 Step 6 使用 STRIP_NONPOLYMER=False 并重新运行。"
    )

if not HOTSPOTS.strip():
    raise ValueError(
        "HOTSPOTS 目前是空的。请先确定 SAP 上希望 binder 接触的表位，"
        "再按格式填写，例如 B230-233,B235-236。"
        "这里的数字只是格式示例，不能直接照抄。"
    )

# 自动使用输入文件中 B 链实际的最小和最大残基编号
res_lo = int(target_res_ids.min())
res_hi = int(target_res_ids.max())
CONTIG = f"{TARGET_CHAIN}{res_lo}-{res_hi},/0,{BINDER_LENGTH}"

spec = {
    "input": INPUT_PDB,
    "contig": CONTIG,
    "ligand": LIGANDS.strip(),
    "allow_ligand_on_existing_chain": True,
    "select_hotspots": HOTSPOTS.strip(),
    "is_non_loopy": IS_NON_LOOPY,
}

if INFER_ORI != "none":
    spec["infer_ori_strategy"] = INFER_ORI

if USE_SYMMETRY:
    spec["symmetry"] = {
        "id": SYM_ID,
        "is_symmetric_motif": True
    }

WORK = globals().get("WORK", "/content/run")
os.makedirs(WORK, exist_ok=True)

IN_JSON = f"{WORK}/input.json"
inp = {DESIGN_NAME: spec}

with open(IN_JSON, "w") as f:
    json.dump(inp, f, indent=2)

print("Input PDB :", INPUT_PDB)
print("Target    :", f"chain {TARGET_CHAIN}, residues {res_lo}-{res_hi}")
print("Contig    :", CONTIG)
print("Ligands   :", LIGANDS)
print("Hotspots  :", HOTSPOTS)
print("Wrote     :", IN_JSON)
print()
print(json.dumps(inp, indent=2))

Input PDB : /content/drive/MyDrive/RFD3/runs/20261026-130058_1SHQ_test_02_261026/target_prepared.pdb
Target    : chain B, residues 1-476
Contig    : B1-476,/0,60-100
Ligands   : ZN,MG
Hotspots  : B120,B122,B123,B160,B172
Wrote     : /content/drive/MyDrive/RFD3/runs/20261026-130058_1SHQ_test_02_261026/input.json

{
  "1SHQ_test_02_261026": {
    "input": "/content/drive/MyDrive/RFD3/runs/20261026-130058_1SHQ_test_02_261026/target_prepared.pdb",
    "contig": "B1-476,/0,60-100",
    "ligand": "ZN,MG",
    "select_hotspots": "B120,B122,B123,B160,B172",
    "is_non_loopy": true,
    "infer_ori_strategy": "hotspots"
  }
}


## 7 (advanced) · Paste raw YAML input  *(optional — overrides the form)*

Full manual control of the RFD3 spec. Edit the YAML below and **run this cell instead of the
form cell above** — it writes exactly this spec to `input.json`, then continue to step 8.

Exposes every RFD3 field the form can't: atom-level `select_hotspots` (`A233: OD1,ND2`), an
explicit **`ori_token`** (a fixed 3-D centre-of-mass for the designed region — confirmed valid
in the RFD3 enzyme tutorial), whole-assembly `length`, `select_fixed_atoms`, `dialect`, etc.

**Workflow for an `ori_token` run (like the NRC2 NB-cap plan below):**
- The `ori_token` is in your input PDB's coordinate frame, so the input **must keep the deposited
  coordinates** — **skip step 5** (idealisation moves/rotates coords and would invalidate the token).
- Prepare the 6-chain target with **step 6** from the raw file: `KEEP_CHAINS = A,B,C,D,E,F`,
  `RES_LO,RES_HI = 200,260`, `STRIP_NONPOLYMER = True` (step 6 only masks atoms, so it preserves
  the deposited frame). That produces `target_prepared.pdb` for the `input:` field below.
- In **step 8**, keep `USE_SYMMETRY = False` (this spec has no `symmetry:` block).
- Multiple top-level entries run as separate jobs in one pass (the three below run together).

⚠️ `plddt_enhanced` is **not** a field I could verify in the RFD3 docs. It's included because
your plan used it — but if `prevalidate_inputs` rejects it, delete that line and rerun.

In [ ]:
#Advanced YAML 是普通 Step 7 表单的高级替代入口：当表单没有你需要的 RFD3 参数时，可以在 YAML 里直接写完整配置。
#这格运行后会把 YAML 转成 input.json，供后续 Step 8 使用。
#例如，普通表单没有提供某些高级约束时，可以尝试在 YAML 中填写；
#RFD3 官方文档列出的输入字段包括 ligand、unindex 和 select_fixed_atoms。

# ADVANCED: raw YAML -> input.json  (edit freely; run this cell to override the form)
import json, os, yaml
WORK = globals().get("WORK", "/content/run"); os.makedirs(WORK, exist_ok=True)

# Prepared target (deposited frame). Upload it to Drive and point here. Reused by all jobs.
TARGET_PDB = "/content/drive/MyDrive/RFD3/targets/target_structure.pdb"

# __WORK__ is replaced with this run's output dir. Point `input:` at your prepared
# 6-chain, deposited-frame target (from step 6). Replace with absolute paths if you prefer.
YAML_INPUT = r"""
job_name:
  input: __TARGET__
paste the yaml input here
"""

YAML_INPUT = YAML_INPUT.replace("__TARGET__", TARGET_PDB)
spec = yaml.safe_load(YAML_INPUT)
assert isinstance(spec, dict) and spec, "Top level must be a mapping: design_name: { ... }"
for name, body in spec.items():
    if not isinstance(body, dict):
        raise ValueError(f"'{name}' must map to a block of fields, got {type(body).__name__}")
    inp = body.get("input")
    if inp and not os.path.exists(inp):
        print(f"WARNING: input path does not exist yet: {inp}")

IN_JSON = f"{WORK}/input.json"
with open(IN_JSON, "w") as f:
    json.dump(spec, f, indent=2)
print(f"Wrote {IN_JSON}  ({len(spec)} job(s): {', '.join(spec)})\n")
print(json.dumps(spec, indent=2))


ScannerError: while scanning a simple key
  in "<unicode string>", line 4, column 1:
    paste the yaml input here
    ^
could not find expected ':'
  in "<unicode string>", line 5, column 1:
    
    ^

## 8 · Run RFD3 (backbone generation)

For symmetry, `inference_sampler.kind=symmetry` and `diffusion_batch_size=1` are required;
add `low_memory_mode=True` on a T4. `n_batches` × `diffusion_batch_size` = total designs
(symmetry forces batch size 1, so scale designs with `n_batches`). `num_timesteps=200` is the
default — drop it to ~50 for a fast smoke test, raise for quality.


In [11]:
#@title RFD3 design
import os
N_BATCHES      = 1     #@param {type:"integer"}   # total designs = N_BATCHES x DIFFUSION_BATCH_SIZE
NUM_TIMESTEPS  = 50   #@param {type:"integer"}
STEP_SCALE     = 1.5   #@param {type:"number"},
LOW_MEMORY     = True  #@param {type:"boolean"}
USE_SYMMETRY   = False #@param {type:"boolean"}
DIFFUSION_BATCH_SIZE = 1 #@param {type:"integer"}  # designs per forward pass (non-symmetric); lower if OOM
SKIP_EXISTING  = False  #@param {type:"boolean"}   # True = resume/top-up, keep designs already in rfd3_out

WORK = globals().get("WORK", "/content/run")
RFD3_OUT = f"{WORK}/rfd3_out"
os.environ.setdefault("FOUNDRY_CHECKPOINT_DIRS", "/content/foundry_ckpts")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # reduce fragmentation OOMs

INPUT_JSON = f"{WORK}/input.json"
cmd = [
    "rfd3 design",
    f"out_dir={RFD3_OUT}",
    f"inputs={INPUT_JSON}",
    "prevalidate_inputs=True",
    f"skip_existing={SKIP_EXISTING}",
    f"n_batches={N_BATCHES}",
    f"diffusion_batch_size={1 if USE_SYMMETRY else DIFFUSION_BATCH_SIZE}",
    f"inference_sampler.num_timesteps={NUM_TIMESTEPS}",
    f"inference_sampler.step_scale={STEP_SCALE}",
    f"low_memory_mode={LOW_MEMORY}",
]
if USE_SYMMETRY:
    cmd.append("inference_sampler.kind=symmetry")

_bs = 1 if USE_SYMMETRY else DIFFUSION_BATCH_SIZE
print(f"Requesting {N_BATCHES} x {_bs} = {N_BATCHES*_bs} design(s) per job"
      + ("; existing designs kept (SKIP_EXISTING)" if SKIP_EXISTING else "") + "\n")
full = " ".join(cmd)
print(full, "\n")
get_ipython().system(full)

import glob
backbones = sorted(glob.glob(f"{RFD3_OUT}/**/*.cif.gz", recursive=True))
print(f"\n{len(backbones)} backbone(s):")
for b in backbones: print(" ", b)


Requesting 1 x 1 = 1 design(s) per job

rfd3 design out_dir=/content/drive/MyDrive/RFD3/runs/20261026-130058_1SHQ_test_02_261026/rfd3_out inputs=/content/drive/MyDrive/RFD3/runs/20261026-130058_1SHQ_test_02_261026/input.json prevalidate_inputs=True skip_existing=False n_batches=1 diffusion_batch_size=1 inference_sampler.num_timesteps=50 inference_sampler.step_scale=1.5 low_memory_mode=True 

/usr/local/lib/python3.13/dist-packages/hydra/_internal/config_loader_impl.py:216: UserWarning: provider=hydra.searchpath in main, path=rfd3.configs is not available.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/hydra/_internal/config_loader_impl.py:216: UserWarning: provider=hydra.searchpath in main, path=configs is not available.
  warnings.warn(
Environment variable CCD_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell

## 9 · LigandMPNN (sequence design)

Redesign each RFD3 backbone's sequence. Unlike `rfd3`/`rf3`, `mpnn` does **not** auto-discover
its weights — it requires an explicit `--checkpoint_path` plus `--is_legacy_weights`
(the cell auto-locates the checkpoint under `FOUNDRY_CHECKPOINT_DIRS`).

Two controls that matter for binder design:
- **`FIXED_CHAINS`** — keep your target chains fixed so MPNN only designs the binder, not the
  target's real sequence. Set this to your target chain IDs (e.g. `A,D`).
- **`TIE_CHAINS`** *(symmetric runs only)* — list the binder subunit chains to tie via
  `--homo_oligomer_chains`, so all subunits get one shared sequence (required for a symmetric
  cap; otherwise each subunit is designed independently and the symmetry breaks).

`--is_legacy_weights` defaults to `True` (matches the Foundry tutorial for installed weights);
flip to `False` if MPNN errors or the sequences look wrong.


In [ ]:
#@title LigandMPNN over all backbones
import glob, os, subprocess, json as _json
MODEL_TYPE     = "ligand_mpnn"  #@param ["ligand_mpnn", "protein_mpnn"]
IS_LEGACY      = True           #@param {type:"boolean"}
FIXED_CHAINS   = ""             #@param {type:"string"}
TIE_CHAINS     = ""             #@param {type:"string"}
N_SEQUENCES    = 4              #@param {type:"integer"}
TEMPERATURE    = 0.1            #@param {type:"number"}
SKIP_EXISTING  = True           #@param {type:"boolean"}   # skip backbones already designed
# FIXED_CHAINS: target chains to keep fixed, e.g. "A,D"  (design only the binder)
# TIE_CHAINS  : symmetric runs only — binder subunit chains to tie to one sequence, e.g. "G,H,I,J,K,L"

CKPT_DIR = os.environ.get("FOUNDRY_CHECKPOINT_DIRS", "/content/foundry_ckpts")
WORK     = globals().get("WORK", "/content/run")
RFD3_OUT = f"{WORK}/rfd3_out"
MPNN_OUT = f"{WORK}/mpnn_out"; os.makedirs(MPNN_OUT, exist_ok=True)

# mpnn needs an explicit checkpoint path — locate it
ckpts = [p for p in glob.glob(f"{CKPT_DIR}/**/*", recursive=True)
         if os.path.isfile(p) and "mpnn" in p.lower()
         and p.endswith((".pt", ".ckpt", ".pth", ".safetensors"))]
assert ckpts, f"No MPNN checkpoint under {CKPT_DIR}; run step 3. Found nothing matching *mpnn*."
# pick the checkpoint matching MODEL_TYPE (a ligand checkpoint with protein_mpnn mismatches)
_want_ligand = (MODEL_TYPE == "ligand_mpnn")
MPNN_CKPT = sorted(ckpts, key=lambda p: (("ligand" in os.path.basename(p).lower()) != _want_ligand))[0]
if ("ligand" in os.path.basename(MPNN_CKPT).lower()) != _want_ligand:
    print(f"WARNING: no checkpoint matching {MODEL_TYPE}; using {os.path.basename(MPNN_CKPT)}")
print("MPNN checkpoint:", MPNN_CKPT)

backbones = (sorted(glob.glob(f"{RFD3_OUT}/**/*.cif.gz", recursive=True))
             or sorted(glob.glob(f"{RFD3_OUT}/**/*.cif", recursive=True))
             or sorted(glob.glob(f"{RFD3_OUT}/**/*.pdb", recursive=True)))
if SKIP_EXISTING:
    _done = [os.path.basename(p) for p in glob.glob(f"{MPNN_OUT}/**/*.cif", recursive=True)]
    _todo = [b for b in backbones
             if not any(n.startswith(os.path.basename(b)[:-3] + "_") for n in _done)]
    print(f"{len(backbones)} backbone(s); {len(backbones)-len(_todo)} already designed, {len(_todo)} to do\n")
    backbones = _todo
else:
    print(f"{len(backbones)} backbone(s)\n")

_failures = []
for _i, bb in enumerate(backbones):
    cmd = ["mpnn",
           "--model_type", MODEL_TYPE,
           "--checkpoint_path", MPNN_CKPT,
           "--is_legacy_weights", str(IS_LEGACY),
           "--structure_path", bb,
           "--out_directory", MPNN_OUT,
           "--batch_size", str(N_SEQUENCES),
           "--temperature", str(TEMPERATURE)]
    if FIXED_CHAINS.strip():
        cmd += ["--fixed_chains", FIXED_CHAINS.strip()]
    if TIE_CHAINS.strip():
        chains = [c.strip() for c in TIE_CHAINS.split(",") if c.strip()]
        cmd += ["--homo_oligomer_chains", _json.dumps([chains])]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(os.path.basename(bb), "->", "ok" if r.returncode == 0 else f"FAIL rc={r.returncode}")
    if r.returncode != 0:
        _failures.append(os.path.basename(bb))
        print(r.stderr[-1500:])
        if _i == 0:        # first one failing is almost always a config error - stop early
            print("\nFirst backbone failed; stopping (likely a settings problem, not a bad file).")
            break
if _failures:
    print(f"\n{len(_failures)} backbone(s) failed: {_failures[:5]}{' ...' if len(_failures)>5 else ''}")

# make MPNN CIFs strict-mmCIF compliant so ChimeraX/PyMOL can open them
# (MPNN writes bare [] metadata values, which strict parsers reject)
import re as _re
for _f in glob.glob(f"{MPNN_OUT}/**/*.cif", recursive=True):
    _t = open(_f).read()
    _t2 = _re.sub(r'(\n_\S+[ \t]+)\[\][ \t]*(?=\n)', r"\1'[]'", _t)
    if _t2 != _t:
        open(_f, "w").write(_t2)

cifs = sorted(glob.glob(f"{MPNN_OUT}/**/*.cif", recursive=True))
fas  = sorted(glob.glob(f"{MPNN_OUT}/**/*.fa*", recursive=True))
print(f"\n{len(cifs)} CIF + {len(fas)} FASTA in {MPNN_OUT}")


# --- export all designed BINDER sequences to one FASTA (binder = chains not in FIXED_CHAINS) ---
import numpy as _np, biotite.structure as _struc, biotite.structure.io as _sio
from collections import Counter as _C
_T2O = {"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
        "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
        "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M","SEC":"U"}
_target = set(c.strip() for c in FIXED_CHAINS.split(",") if c.strip())
_records = []
for _cif in sorted(glob.glob(f"{MPNN_OUT}/**/*.cif", recursive=True)):
    try:
        _a = _sio.load_structure(_cif); _a = _a[_struc.filter_amino_acids(_a)]
    except Exception:
        continue
    _chains = sorted(set(_a.chain_id.tolist()))
    if _target:
        _binders = [c for c in _chains if c not in _target]
    else:
        _cnt = {c: len(_np.unique(_a.res_id[_a.chain_id==c])) for c in _chains}
        _mode = _C(_cnt.values()).most_common(1)[0][0]
        _binders = [c for c,n in _cnt.items() if n != _mode]
    _stem = _cif.split("/")[-1].replace(".cif","")
    _seen = set()
    for _ch in _binders:
        _m = (_a.chain_id==_ch) & (_a.atom_name=="CA")
        _seq = "".join(_T2O.get(rn,"X") for rn in _a.res_name[_m][_np.argsort(_a.res_id[_m])])
        if _seq and _seq not in _seen:
            _seen.add(_seq); _records.append((f"{_stem}|chain_{_ch}|len_{len(_seq)}", _seq))
_FASTA = f"{MPNN_OUT}/binder_sequences.fasta"
with open(_FASTA, "w") as _fh:
    for _h, _s in _records:
        _fh.write(f">{_h}\n{_s}\n")
print(f"{len(_records)} binder sequence(s) written to {_FASTA}")


## 10 · RF3 (refold / validation)

Refold each MPNN-designed structure with RF3 and keep its confidence metrics — this is your
AF3-style designability check. Pass **absolute paths** to the input CIFs.


In [ ]:
#@title RF3 fold
import glob, os, subprocess
NUM_STEPS     = 200   #@param {type:"integer"}
SKIP_EXISTING = True  #@param {type:"boolean"}   # skip designs already refolded

WORK = globals().get("WORK", "/content/run")
MPNN_OUT = f"{WORK}/mpnn_out"
RF3_OUT  = f"{WORK}/rf3_out"; os.makedirs(RF3_OUT, exist_ok=True)
os.environ.setdefault("FOUNDRY_CHECKPOINT_DIRS", "/content/foundry_ckpts")

def _has_output(stem, names):
    """True if any output file belongs to `stem`. Requires the next character after the
    stem to be a non-digit so d1 is not matched by a d10 output."""
    for n in names:
        if n.startswith(stem) and (len(n) == len(stem) or not n[len(stem)].isdigit()):
            return True
    return False

cifs = sorted(glob.glob(f"{MPNN_OUT}/**/*.cif", recursive=True))
if SKIP_EXISTING:
    _done = [os.path.basename(p) for p in glob.glob(f"{RF3_OUT}/**/*", recursive=True)
             if os.path.isfile(p)]
    _todo = [c for c in cifs if not _has_output(os.path.basename(c)[:-4], _done)]
    print(f"{len(cifs)} design(s); {len(cifs)-len(_todo)} already refolded, {len(_todo)} to do\n")
    cifs = _todo
else:
    print(f"{len(cifs)} MPNN structure(s) to refold\n")

_failures = []
for _i, c in enumerate(cifs):
    cmd = ["rf3", "fold",
           f"inputs={os.path.abspath(c)}",
           f"out_dir={RF3_OUT}",
           f"num_steps={NUM_STEPS}",
           "diffusion_batch_size=1"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(os.path.basename(c), "->", "ok" if r.returncode == 0 else f"FAIL rc={r.returncode}")
    if r.returncode != 0:
        _failures.append(os.path.basename(c))
        print(r.stderr[-1500:])
        if _i == 0:
            print("\nFirst design failed; stopping (likely a settings problem, not a bad file).")
            break
if _failures:
    print(f"\n{len(_failures)} design(s) failed: {_failures[:5]}{' ...' if len(_failures)>5 else ''}")

preds = sorted(glob.glob(f"{RF3_OUT}/**/*.cif*", recursive=True))
print(f"\n{len(preds)} RF3 prediction(s) in {RF3_OUT}")


## 11 · Collect metrics & rank designs

Pulls confidence fields (pLDDT / pTM / pAE-type) from the RF3 sidecar JSONs and tabulates
them. Field names vary by version, so this scans each JSON for likely keys rather than
assuming a fixed schema. For a true binder filter you'll add interface metrics
(interface pAE / ipTM, BSA, contact counts) — hooks noted in the cell.


In [ ]:
#@title Build metrics table
import glob, json, os, pandas as pd

WORK = globals().get("WORK", "/content/run")
RF3_OUT = f"{WORK}/rf3_out"
jsons = sorted(glob.glob(f"{RF3_OUT}/**/*.json", recursive=True))

WANT = ["plddt", "mean_plddt", "ptm", "iptm", "pae", "mean_pae", "confidence"]
rows = []
for j in jsons:
    try:
        d = json.load(open(j))
    except Exception:
        continue
    flat = {}
    def walk(prefix, obj):
        if isinstance(obj, dict):
            for k, v in obj.items(): walk(f"{prefix}{k}.", v)
        elif isinstance(obj, (int, float)):
            flat[prefix[:-1]] = obj
    walk("", d)
    row = {"design": os.path.basename(j)}
    for k, v in flat.items():
        if any(w in k.lower() for w in WANT):
            row[k] = v
    rows.append(row)

df = pd.DataFrame(rows)
if not df.empty:
    sort_col = next((c for c in df.columns if "plddt" in c.lower()), None)
    if sort_col: df = df.sort_values(sort_col, ascending=False)
    df.to_csv(f"{WORK}/design_metrics.csv", index=False)
print(df.head(20).to_string() if not df.empty else "No JSON metrics parsed — inspect rf3_out/ manually.")

# --- TODO: interface metrics for binder filtering -----------------------------
# Compute on the RF3 complex: interface pAE (binder<->target blocks of the PAE matrix),
# buried surface area (biotite SASA: complex vs isolated chains), and Cα contact counts
# within 8 Å across the interface. These map onto the ipSAE/BSA filters you already use.


## 12 · Download results

In [ ]:
#@title Zip & download outputs
import shutil, os
from google.colab import files
WORK = globals().get("WORK", "/content/run")

if WORK.startswith("/content/drive/"):
    print("Outputs already persisted on Drive at:", WORK, "\n(no download needed)")
else:
    shutil.make_archive("/content/rfd3_run", "zip", WORK)
    files.download("/content/rfd3_run.zip")


---
### Notes & next steps
- **Symmetry sanity check.** If you ran symmetric, open a backbone in PyMOL/ChimeraX and
  confirm it's genuinely Cn/Dn and seated on your hotspots. Drift off-axis usually means
  revisiting step 5 (alignment / idealisation) before touching diffusion params.
- **Designability loop.** The real filter is RF3-refold vs RFD3-backbone agreement (Cα-RMSD)
  plus interface confidence — wire the RMSD + interface-pAE/BSA hooks in step 11 into your
  filtering.
- **Throughput.** Symmetric runs are batch-1; scale with `n_batches` or move off the T4. For
  large screens this is better as a SLURM array than Colab.
- **Docs:** [input spec](https://rosettacommons.github.io/foundry/models/rfd3/input.html) ·
  [symmetry](https://rosettacommons.github.io/foundry/models/rfd3/examples/symmetry.html) ·
  [PPI tutorial](https://rosettacommons.github.io/foundry/models/rfd3/tutorials/ppi_design_tutorial.html) ·
  [Foundry repo](https://github.com/RosettaCommons/foundry).
